# Day 25：vLLM 服务安全与模型部署加固

🟠 Agent 安全与部署运维 · 第 4 周

[在 GitHub 查看教程](https://github.com/Siebelyk/fde-daily-plan/blob/main/tutorials/Day-25.md)

## 学习目标

1. 理解 vLLM 的工作原理和部署架构
2. 理解模型部署的安全风险
3. 实现 vLLM 安全加固配置


## 推荐资料

- 📖 文档 [vLLM Documentation](https://docs.vllm.ai/)
- 🎬 视频 [vLLM Deployment Tutorial](https://www.youtube.com/watch?v=2r5v8e3k4pZ)
- 🔧 工具 [TGI (Text Generation Inference)](https://huggingface.co/docs/text-generation-inference/)


## Demo：vLLM 安全部署实验：从默认配置到安全加固

搭建 vLLM 推理服务，从默认配置开始逐步加固安全配置，对比加固前后的安全表现

难度：进阶 | 预计：2h

## 推荐练习方式：🐳 DevOps 实操

## 推荐练习方式：用 Docker 实际部署 vLLM 并加固

> vLLM 是推理引擎，学习它的安全应该是实际部署和配置，不是写 Python wrapper。

### 步骤

1. **用 Docker 部署 vLLM**（需要 GPU 或 CPU-only 模式）：
   ```bash
   # GPU 模式
   docker run --gpus all -p 8000:8000 \
     vllm/vllm-openai:latest \
     --model meta-llama/Llama-3.2-1B-Instruct \
     --trust-remote-code

   # CPU 模式（无 GPU 也能学）
   docker run -p 8000:8000 \
     vllm/vllm-openai:latest \
     --model meta-llama/Llama-3.2-1B-Instruct \
     --device cpu
   ```

2. **测试默认配置的安全性**：
   ```bash
   # 测试 1: 是否需要认证？
   curl http://localhost:8000/v1/chat/completions \
     -H "Content-Type: application/json" \
     -d '{"model":"meta-llama/Llama-3.2-1B-Instruct","messages":[{"role":"user","content":"hi"}]}'
   # 如果直接返回结果 → 无认证！

   # 测试 2: 是否有速率限制？
   for i in $(seq 1 100); do curl -s http://localhost:8000/v1/models; done
   # 如果全部成功 → 无限流！

   # 测试 3: 是否泄露模型信息？
   curl http://localhost:8000/v1/models
   ```

3. **逐步加固配置**：
   ```bash
   # 加固 1: 添加 API Key 认证
   docker run -p 8000:8000 \
     -e VLLM_API_KEY=your-secret-key \
     vllm/vllm-openai:latest \
     --model meta-llama/Llama-3.2-1B-Instruct \
     --api-key your-secret-key

   # 加固 2: 限制最大 token 数
   --max-num-seqs 4 --max-model-len 2048

   # 加固 3: 禁用前缀缓存（防止缓存投毒）
   --no-enable-prefix-caching
   ```

4. **加固后重测**：
   - 认证是否生效？（无 key 请求应被拒绝）
   - 限流是否生效？（连续请求应被 429）
   - 记录加固前后的安全对比表

### 为什么不写代码？
vLLM 安全的核心是**配置和部署**，不是编程。实际用 Docker 跑一遍、改参数、
测效果，比看 Python 模拟脚本更贴近真实工作场景。代码版本作为附录保留。


---

## 附录：代码参考

> 以下为 Python 代码实现，作为推荐练习方式的补充参考。

## 环境准备


In [ ]:
# 安装 vLLM（需要 GPU）
!pip install vllm
# 或用 Docker: docker run --gpus all vllm/vllm-openai:latest --model meta-llama/Llama-2-7b-chat-hf




## 原理速览
vLLM = 高性能 LLM 推理引擎，兼容 OpenAI API。
默认配置的安全风险：
1. 无认证：任何人都能访问 API
2. 无速率限制：容易被 DDoS
3. 无输出过滤：可能输出有害内容
4. 日志不足：无法追溯攻击

## 配置对比

### 不安全配置（默认）


In [ ]:
# ❌ 不安全：无认证、无限流、无日志
!python -m vllm.entrypoints.openai.api_server   --model meta-llama/Llama-2-7b-chat-hf   --port 8000




### 安全配置（加固后）


In [ ]:
# ✅ 安全：认证 + 限流 + 过滤 + 日志
!python -m vllm.entrypoints.openai.api_server   --model meta-llama/Llama-2-7b-chat-hf   --port 8000   --api-key YOUR_SECURE_API_KEY   --max-num-seqs 64   --max-model-len 4096   --disable-log-requests   --chat-template ./safe_template.jinja




## 代码：安全配置验证脚本


In [ ]:
import requests, json, subprocess, time

VLLM_URL = "http://localhost:8000"
API_KEY = "YOUR_SECURE_API_KEY"

def check_security_config():
    """检查 vLLM 实例的安全配置"""
    checks = []

    # 1. 认证检查
    try:
        r = requests.post(f"{VLLM_URL}/v1/chat/completions",
                         json={"model": "meta-llama/Llama-2-7b-chat-hf",
                               "messages": [{"role": "user", "content": "hi"}]})
        if r.status_code == 401:
            checks.append(("API Auth", True, "Unauthorized without key"))
        else:
            checks.append(("API Auth", False, f"No auth required (status={r.status_code})"))
    except:
        checks.append(("API Auth", False, "Cannot connect"))

    # 2. 带认证的请求
    headers = {"Authorization": f"Bearer {API_KEY}"}
    try:
        r = requests.post(f"{VLLM_URL}/v1/chat/completions",
                         headers=headers,
                         json={"model": "meta-llama/Llama-2-7b-chat-hf",
                               "messages": [{"role": "user", "content": "hi"}],
                               "max_tokens": 10})
        checks.append(("API Call", r.status_code == 200, f"Status: {r.status_code}"))
    except Exception as e:
        checks.append(("API Call", False, str(e)[:50]))

    # 3. 速率限制检查
    responses = []
    for i in range(20):
        r = requests.post(f"{VLLM_URL}/v1/chat/completions",
                         headers=headers,
                         json={"model": "meta-llama/Llama-2-7b-chat-hf",
                               "messages": [{"role": "user", "content": "hi"}],
                               "max_tokens": 5})
        responses.append(r.status_code)
    unique = set(responses)
    if 429 in unique:
        checks.append(("Rate Limit", True, "429 returned"))
    else:
        checks.append(("Rate Limit", False, "No rate limiting"))

    # 4. 模型信息泄露
    try:
        r = requests.get(f"{VLLM_URL}/v1/models", headers=headers)
        models = r.json()
        if "data" in models and len(models["data"]) > 0:
            checks.append(("Model Info", False, f"Model list exposed: {[m['id'] for m in models['data']]}"))
        else:
            checks.append(("Model Info", True, "Model list hidden"))
    except:
        checks.append(("Model Info", True, "Cannot access"))

    print("=== vLLM Security Config Check ===
")
    for name, passed, detail in checks:
        status = "PASS" if passed else "FAIL"
        print(f"  [{status:4s}] {name:15s} {detail}")

# 运行检查（需要 vLLM 在 localhost:8000 运行）
# check_security_config()




## 安全分析
vLLM 部署安全 = API 认证 + 速率限制 + 输出过滤 + 日志审计 + 资源隔离。生产环境建议加上反向代理（nginx/traefik）做 TLS 和 WAF。

## 进阶挑战

1. 用 Docker Compose 部署 vLLM + nginx + Redis 做完整安全栈
   - 思路提示：docker-compose.yml 定义 vllm + nginx（反向代理 + 限流） + redis（缓存/限流） 三层架构
   - 参考：[vLLM 部署文档](https://docs.vllm.ai/en/latest/serving/deployment.html)
2. 研究 vLLM 的 PagedAttention 对安全的影响
   - 思路提示：PagedAttention 的分页复用在多用户场景下可能带来跨 session 信息泄露，需测试隔离性
   - 参考：[vLLM / PagedAttention 论文](https://arxiv.org/abs/2309.06180)
3. 实现 vLLM 的 Prometheus 指标导出
   - 思路提示：vLLM 内置 Prometheus 指标，查看 /metrics 端点；关注 num_requests_running、gpu_cache_usage
   - 参考：[vLLM Metrics 文档](https://docs.vllm.ai/en/latest/serving/metrics.html)


---

## 明日预告

**Day 26：容器化 LLM 服务安全**
🟠 Agent 安全与部署运维 · 第 4 周